# Financial Trend Analysis: Microsoft, Tesla and Apple (FY2023 to FY2025)

**Goal:** Extract key figures from each company's 10-K filings and analyse year-over-year trends, to inform the design of an AI-powered financial chatbot.

**Companies:** Microsoft, Tesla, Apple  
**Metrics:** Total Revenue, Net Income, Total Assets, Total Liabilities, Cash Flow from Operating Activities  
**Source:** Each company's annual 10-K filing on SEC EDGAR (Income Statement, Balance Sheet and Cash Flow Statement).

## Methodology and data notes
1. Figures were extracted manually from the 10-K filings and compiled in a spreadsheet (`financials.csv`).
2. **All values are in USD millions.**
3. **Fiscal years differ by company:** Microsoft's year ends 30 June, Apple's in late September, Tesla's on 31 December. Each company is analysed on its own fiscal year, so a given "2025" is not the same calendar period for all three.
4. **Revenue labels differ:** Apple reports "Total net sales", Microsoft "Total revenue", Tesla "Total revenues". All are treated as Total Revenue.
5. **Tesla net income:** I use *net income attributable to common stockholders* (14,997 / 7,091 / 3,794). Tesla's headline "net income" including non-controlling interests is slightly different (14,974 / 7,153 / 3,855). The conclusions do not change.
6. Year-over-year (YoY) growth is `(this year - last year) / last year * 100`, calculated within each company. The first year of each company has no previous year, so its growth is `NaN` by design.

## Step 1: Import libraries

In [1]:
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## Step 2: Load the extracted data
The data is one row per company per fiscal year.

In [2]:
df = pd.read_csv('financials.csv')
df

,Company,Fiscal Year,Total Revenue,Net Income,Total Assets,Total Liabilities,Operating Cash Flow
0,Apple,2023,383285,96995,352583,290437,110543
1,Apple,2024,391035,93736,364980,308030,118254
2,Apple,2025,416161,112010,359241,285508,111482
3,Microsoft,2023,211915,72361,411976,205753,87582
4,Microsoft,2024,245122,88136,512163,243686,118548
5,Microsoft,2025,281724,101832,619003,275524,136162
6,Tesla,2023,96773,14997,106618,43009,13256
7,Tesla,2024,97690,7091,122070,48390,14923
8,Tesla,2025,94827,3794,137806,54941,14747


### Data quality checks
Before analysing, confirm there are no missing values, the expected 9 rows (3 companies x 3 years), and that liabilities are always smaller than assets for these companies.

In [3]:
print('Rows:', len(df), '| Missing values:', int(df.isna().sum().sum()))
print('Rows per company:')
print(df['Company'].value_counts().to_string())
assert (df['Total Liabilities'] < df['Total Assets']).all(), 'Liabilities should be below assets'
print('Checks passed')

Rows: 9 | Missing values: 0
Rows per company:
Company
Apple        3
Microsoft    3
Tesla        3
Checks passed


## Step 3: Year-over-year growth
The data must be sorted by company and year first, because `pct_change()` compares each row with the row above it. `groupby('Company')` makes sure Microsoft is never compared with Tesla.

In [4]:
df = df.sort_values(['Company', 'Fiscal Year']).reset_index(drop=True)

df['Revenue Growth (%)'] = df.groupby('Company')['Total Revenue'].pct_change() * 100
df['Net Income Growth (%)'] = df.groupby('Company')['Net Income'].pct_change() * 100
df['Assets Growth (%)'] = df.groupby('Company')['Total Assets'].pct_change() * 100
df['Liabilities Growth (%)'] = df.groupby('Company')['Total Liabilities'].pct_change() * 100
df['Operating Cash Flow Growth (%)'] = df.groupby('Company')['Operating Cash Flow'].pct_change() * 100

growth_cols = [c for c in df.columns if 'Growth' in c]
df[['Company', 'Fiscal Year'] + growth_cols].round(1)

,Company,Fiscal Year,Revenue Growth (%),Net Income Growth (%),Assets Growth (%),Liabilities Growth (%),Operating Cash Flow Growth (%)
0,Apple,2023,NaN,NaN,NaN,NaN,NaN
1,Apple,2024,2.0,-3.4,3.5,6.1,7.0
2,Apple,2025,6.4,19.5,-1.6,-7.3,-5.7
3,Microsoft,2023,NaN,NaN,NaN,NaN,NaN
4,Microsoft,2024,15.7,21.8,24.3,18.4,35.4
5,Microsoft,2025,14.9,15.5,20.9,13.1,14.9
6,Tesla,2023,NaN,NaN,NaN,NaN,NaN
7,Tesla,2024,0.9,-52.7,14.5,12.5,12.6
8,Tesla,2025,-2.9,-46.5,12.9,13.5,-1.2


## Step 4: Further analysis
To compare the companies on a like-for-like basis, I calculate the average growth per company plus three ratios:
- **Net profit margin** = Net Income / Total Revenue
- **Liabilities-to-assets** = Total Liabilities / Total Assets (how heavily the company is financed by obligations)
- **Operating cash flow / net income** = how much real cash the company generates per dollar of reported profit

In [5]:
avg_growth = df.groupby('Company')[growth_cols].mean().round(1)
avg_growth

,Revenue Growth (%),Net Income Growth (%),Assets Growth (%),Liabilities Growth (%),Operating Cash Flow Growth (%)
Company,,,,,
Apple,4.2,8.1,1.0,-0.6,0.6
Microsoft,15.3,18.7,22.6,15.8,25.1
Tesla,-1.0,-49.6,13.7,13.0,5.7


In [6]:
df['Net Profit Margin (%)'] = df['Net Income'] / df['Total Revenue'] * 100
df['Liabilities to Assets (%)'] = df['Total Liabilities'] / df['Total Assets'] * 100
df['Op. Cash Flow / Net Income (x)'] = df['Operating Cash Flow'] / df['Net Income']

ratio_cols = ['Net Profit Margin (%)', 'Liabilities to Assets (%)', 'Op. Cash Flow / Net Income (x)']
df[['Company', 'Fiscal Year'] + ratio_cols].round(1)

,Company,Fiscal Year,Net Profit Margin (%),Liabilities to Assets (%),Op. Cash Flow / Net Income (x)
0,Apple,2023,25.3,82.4,1.1
1,Apple,2024,24.0,84.4,1.3
2,Apple,2025,26.9,79.5,1.0
3,Microsoft,2023,34.1,49.9,1.2
4,Microsoft,2024,36.0,47.6,1.3
5,Microsoft,2025,36.1,44.5,1.3
6,Tesla,2023,15.5,40.3,0.9
7,Tesla,2024,7.3,39.6,2.1
8,Tesla,2025,4.0,39.9,3.9


In [7]:
# Total change over the whole period, FY2023 -> FY2025
metrics = ['Total Revenue', 'Net Income', 'Total Assets', 'Total Liabilities', 'Operating Cash Flow']
first = df[df['Fiscal Year'] == 2023].set_index('Company')[metrics]
last = df[df['Fiscal Year'] == 2025].set_index('Company')[metrics]
((last / first - 1) * 100).round(1)

,Total Revenue,Net Income,Total Assets,Total Liabilities,Operating Cash Flow
Company,,,,,
Apple,8.6,15.5,1.9,-1.7,0.8
Microsoft,32.9,40.7,50.3,33.9,55.5
Tesla,-2.0,-74.7,29.3,27.7,11.2


## Step 5: Findings

**Microsoft: consistent, profitable growth.** Revenue grew about 15.7% in FY2024 and 14.9% in FY2025, with net income up 21.8% and 15.5%. Net margin stayed high at roughly 34% to 36%. Operating cash flow rose 35.4% then 14.9%. Assets grew faster than liabilities, so liabilities fell from about 50% of assets to about 45%. Over the two years revenue is up about 33% and net income about 41%.

**Apple: stable, with a strong FY2025 rebound.** Revenue growth was modest (2.0% in FY2024, 6.4% in FY2025). Net income dipped 3.4% in FY2024, then jumped 19.5% in FY2025, lifting the margin to about 27%. Total liabilities fell 7.3% in FY2025 and assets slipped 1.6%, so the balance sheet shrank slightly. Apple carries a high liabilities-to-assets ratio (about 80% to 84%) compared with the other two.

**Tesla: growing balance sheet, shrinking profits.** Revenue was nearly flat (+0.9% then -2.9%), while net income fell about 53% and then 47%, and net margin dropped from about 15.5% to 4.0%. Assets and liabilities both grew by roughly 13% a year. Operating cash flow held up (about 13.3bn to 14.7bn) even as net income collapsed, so cash flow was 3.9x net income in FY2025. This suggests large non-cash expenses (such as depreciation) reduced reported profit more than cash generation, which would be worth confirming in the cash flow statement.

**Cross-company takeaways**
- Microsoft is the fastest grower and the most profitable and least leveraged of the three on these measures.
- Apple's profits are the largest in absolute terms in FY2025, but it grows slowly.
- Tesla has the weakest profit trend, although its cash generation is steadier than its earnings.

## Limitations and implications for the chatbot
- Only three years and three companies, so the trends are indicative, not conclusive.
- Different fiscal year-ends mean the years are not perfectly aligned.
- A financial chatbot built on this data could answer questions such as *"What was Apple's net income in 2025?"* or *"How did Tesla's revenue change from 2024 to 2025?"* directly from this table. It should always state the units (USD millions) and the fiscal-year convention, and it should refuse or ask for clarification when a company, metric or year is not in the dataset.